In [1]:
'''
This script is used to generate scenario folders, copy weather files and run simulations using parallel processing.

'''


import os
import shutil
import subprocess
import sys
import multiprocessing
from read_swat import *
from concurrent.futures import ThreadPoolExecutor
from geopy.distance import geodesic

os.chdir('..') # Changing to main SWATPlusForCRIDA

In [2]:
# Functions
'''
This are written as functions as they will be used repeatedly in the script.
But also they will be run in parallel using ThreadPoolExecutor.
'''

def create_folder(folder_name):
    """Create a folder if it doesn't already exist."""
    if not os.path.exists(folder_name):
        os.mkdir(folder_name)
        print(f"Folder '{folder_name}' created.")
    else:
        print(f"Folder '{folder_name}' already exists.")
        
def copy_folder_contents(source_folder, destination_folder):
    """Recursively copy contents from source_folder to destination_folder."""
    if not os.path.exists(source_folder):
        print(f"Source folder '{source_folder}' does not exist.")
        return
    
    if not os.path.exists(destination_folder):
        print(f"Destination folder '{destination_folder}' does not exist. Creating it.")
        create_folder(destination_folder)
    
    for item in os.listdir(source_folder):
        src_path = os.path.join(source_folder, item)
        dest_path = os.path.join(destination_folder, item)
        if os.path.isdir(src_path):
            shutil.copytree(src_path, dest_path, dirs_exist_ok=True)  # Copy directory recursively
            # print(f"Copied directory '{src_path}' to '{dest_path}'.")
        else:
            shutil.copy2(src_path, dest_path)  # Copy file
            # print(f"Copied file '{src_path}' to '{dest_path}'.")
    print (f"Copied contents of {source_folder} to {destination_folder}")

def parallel_copy_jobs(jobs):
    """Execute copy jobs in parallel."""
    with ThreadPoolExecutor(max_workers=14) as executor:
        executor.map(lambda args: copy_folder_contents(*args), jobs)
        
# Functions
def create_cli(stat_list,var,cli_path):
    with open(cli_path, 'w') as f:
        f.write(f"{var}.cli : written by Jose Teran \n")
        f.write(f"filename \n")

        df_cli = stat_list[["NAME"]]
        df_cli = df_cli["NAME"]
        
        df_cli.to_csv(f, index=False, header=None, sep="\t",lineterminator='\n')
         
        print(f"File {cli_path} succesfully saved")

### Copying model folders for all scenarios

In [3]:
# Models, experiments and paths

model_list=["SAM_44_RCA4_EC_EARTH",
            "SAM_44_RCA4_GFDL",
            "SAM_44_RCA4_HadGEM2",
            "SAM_44_RCA4_IPSL",
            "SAM_44_RCA4_MIROC5"]

experiment_list=["historical",
                 "rcp45",
                 "rcp85"]


models_folder = "data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox"
weather_folder = "data/CaseStudy/data/climate_data/RCM/swat_weather"

folder_source_model = "Calibration"

processes = 14 # Number of processes to run in parallel 

In [4]:
# Generating new paths and copying

dest_model_dirs = []
for model in model_list:
    for scenario in experiment_list:
        dest_model_dirs.append(f"{models_folder}/{scenario}_{model}")

source_weather_dirs = []
for model in model_list:
    for scenario in experiment_list:
        source_weather_dirs.append(f"{weather_folder}/{scenario}/{model}")

In [5]:
# Copying SWAT model files
jobs = [(f"{models_folder}/{folder_source_model}", job) for job in dest_model_dirs] # Getting jobs for parallel processing
parallel_copy_jobs(jobs)

# Copying weather files
jobs = [(source, dest) for source, dest in zip(source_weather_dirs, dest_model_dirs)]
parallel_copy_jobs(jobs)

Copied contents of data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/Calibration to data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/historical_SAM_44_RCA4_IPSLCopied contents of data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/Calibration to data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/rcp45_SAM_44_RCA4_HadGEM2

Copied contents of data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/Calibration to data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/rcp85_SAM_44_RCA4_EC_EARTH
Copied contents of data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/Calibration to data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/rcp85_SAM_44_RCA4_HadGEM2
Copied contents of data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/Calibration to data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/rcp85_SAM_44_RCA4_IPSL
Copied contents of data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox

### Adjusting model files

In [ ]:
# Edit weather-station file
weather_sta_cli = "weather-sta.cli"
weather_sta_file = f"{models_folder}/{folder_source_model}/{weather_sta_cli}"


weather_sta_df = swat_ModelFile(weather_sta_file).dframe

# Reading pcp and tmp files, get their names, latitude and longitude

pcp_files = [x for x in os.listdir(f"{models_folder}/{experiment_list[0]}_{model_list[0]}") if x.endswith('.pcp') and "station" in x]
tmp_files = [x for x in os.listdir(f"{models_folder}/{experiment_list[0]}_{model_list[0]}") if x.endswith('.tmp') and "station" in x]

def get_coords_swatStation(list,path):
    
    lats  = []
    lons  = []
    names = []
    
    for file in list:
        file_path = f"{path}/{file}"
        names.append(file)
        # Read text and 
        with open(file_path) as f:
            lines = f.readlines()
            
            lat = float(lines[2].split()[2])
            lon = float(lines[2].split()[3])
        
        lats.append(lat)
        lons.append(lon)            
        
    df = pd.DataFrame({"name":names,"lat":lats,"lon":lons})
    
    return df

def find_closest_station(lat, lon, stations_df):
    min_dist = float('inf')
    closest_station = None
    for _, row in stations_df.iterrows():
        dist = geodesic((lat, lon), (row['lat'], row['lon'])).kilometers
        if dist < min_dist:
            min_dist = dist
            closest_station = row['name']  # or whatever the column is
    return closest_station
    
pcp_df = get_coords_swatStation(pcp_files,f"{models_folder}/{experiment_list[0]}_{model_list[0]}")        
tmp_df = get_coords_swatStation(tmp_files,f"{models_folder}/{experiment_list[0]}_{model_list[0]}")

weather_sta_df['lat'] = None
weather_sta_df['lon'] = None

for index,row in weather_sta_df.iterrows():
    # Get the latitude and longitude from the name string (s1620s68700w means: 16.20°S, 68.70°W)
    lat_str = row['name'][1:5]
    lon_str = row['name'][7:11]
    west_or_east = row['name'][-1]
    north_or_south = row['name'][6]
    
    if west_or_east == 'w':
        lon = -float(lon_str)/100
    else:
        lon = float(lon_str)/100
    if north_or_south == 's':
        lat = -float(lat_str)/100
    else:
        lat = float(lat_str)/100
        
    weather_sta_df.loc[index,'lat'] = lat
    weather_sta_df.loc[index,'lon'] = lon
            
weather_sta_df['pcp'] = weather_sta_df.apply(
    lambda row: find_closest_station(row['lat'], row['lon'], pcp_df), axis=1)

weather_sta_df['tmp'] = weather_sta_df.apply(
    lambda row: find_closest_station(row['lat'], row['lon'], tmp_df), axis=1)

# Fill null columns that were taken as nothing
weather_sta_df["pet"]      = weather_sta_df["pet"].astype(str)
weather_sta_df["atmo_dep"] = weather_sta_df["atmo_dep"].astype(str)
weather_sta_df.loc[:,"pet"]      = "null"
weather_sta_df.loc[:,"atmo_dep"] = "null"

weather_sta_df = weather_sta_df[['name', 'wgn', 'pcp', 'tmp', 'slr', 'hmd', 'wnd', 'pet', 'atmo_dep']]

weather_sta_cli_file = swat_ModelFile(weather_sta_file)
weather_sta_cli_file.dframe = weather_sta_df

weather_sta_cli_file.write("weather-sta.cli updated for RCM scenarios",overwrite=False,write_path=f"{models_folder}/{experiment_list[0]}_{model_list[0]}/weather-sta.cli")

# Copy updated file for rest of scenarios

for model in dest_model_dirs[1:]:
    src_file = f"{models_folder}/{experiment_list[0]}_{model_list[0]}/weather-sta.cli"
    shutil.copy(src_file,model)
    
    print(f"Copied updated weather-sta.cli file to {model}")

In [ ]:
# Update <variable>.cli file for all scenarios
pcp_list = weather_sta_df[['pcp']].drop_duplicates()
tmp_list = weather_sta_df[['tmp']].drop_duplicates()

pcp_list = pcp_list.rename(columns={"pcp":"NAME"})
tmp_list = tmp_list.rename(columns={"tmp":"NAME"})

pcpcli_file = f"{models_folder}/{experiment_list[0]}_{model_list[0]}/pcp.cli"
tmpcli_file = f"{models_folder}/{experiment_list[0]}_{model_list[0]}/tmp.cli"

create_cli(pcp_list,'pcp',pcpcli_file)
create_cli(tmp_list,'tmp',tmpcli_file)

# Copy to the other folders

for model in dest_model_dirs[1:]:
    src_file = f"{models_folder}/{experiment_list[0]}_{model_list[0]}/pcp.cli"
    shutil.copy(src_file,model)
    
    print(f"Copied updated pcp.cli file to {model}")
    
for model in dest_model_dirs[1:]:
    src_file = f"{models_folder}/{experiment_list[0]}_{model_list[0]}/tmp.cli"
    shutil.copy(src_file,model)
    
    print(f"Copied updated tmp.cli file to {model}")        

In [8]:
# Updating sim.time files
for model in dest_model_dirs:
    simtime_file_path = f"{model}/time.sim"
    
    scenario = model.split("/")[-1].split("_")[0]
    
    if scenario == "historical":
        start = 1976
        end   = 2005
        
    else:
        start = 2071
        end   = 2100
    with open(simtime_file_path) as f:
        lines = f.readlines()
        line = lines[2].split()
        
        line[1] = str(start)
        line[3] = str(end)
        lines[2] = '\t'.join(line)
        
    with open(simtime_file_path, 'w') as f:
        f.writelines(lines)
        
    print(f"Written time.sim file for {model}")

Written time.sim file for data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/historical_SAM_44_RCA4_EC_EARTH
Written time.sim file for data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/rcp45_SAM_44_RCA4_EC_EARTH
Written time.sim file for data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/rcp85_SAM_44_RCA4_EC_EARTH
Written time.sim file for data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/historical_SAM_44_RCA4_GFDL
Written time.sim file for data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/rcp45_SAM_44_RCA4_GFDL
Written time.sim file for data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/rcp85_SAM_44_RCA4_GFDL
Written time.sim file for data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/historical_SAM_44_RCA4_HadGEM2
Written time.sim file for data/CaseStudy/Models/katari-swat-crida-spt/scenarios/Toolbox/rcp45_SAM_44_RCA4_HadGEM2
Written time.sim file for data/CaseStudy/Models/katari-swat-crida-spt/scenarios